##Phase 1

In [38]:
from google.colab import drive
import zipfile

# Mount Drive
drive.mount('/content/drive')

# Extract your dataset
zip_path = "/content/drive/MyDrive/Copy of HV-AI-2025.zip"
extract_path = "/content/hv_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracted to: /content/hv_dataset


In [39]:
import os
import pandas as pd
import shutil
from sklearn.model_selection import train_test_split

base_dir = "/content/hv_dataset/HV-AI-2025/labeled_data"
img_dir = os.path.join(base_dir, "images")
csv_path = os.path.join(base_dir, "labeled_data.csv")

df = pd.read_csv(csv_path)

# Train-test split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

# Create folders
split_dir = "/content/hv_dataset/split"
for split in ["train", "test"]:
    for label in df['label'].unique():
        os.makedirs(os.path.join(split_dir, split, label), exist_ok=True)

# Move files to correct folders
def move_images(subset_df, subset):
    for _, row in subset_df.iterrows():
        src = os.path.join(img_dir, row['img_name'])
        dst = os.path.join(split_dir, subset, row['label'], row['img_name'])
        if os.path.exists(src):
            shutil.copy2(src, dst)

move_images(train_df, "train")
move_images(test_df, "test")

print("Train/Test folders prepared at:", split_dir)

Train/Test folders prepared at: /content/hv_dataset/split


In [40]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1, 0.1),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(os.path.join(split_dir, "train"), transform=train_transform)
test_dataset = datasets.ImageFolder(os.path.join(split_dir, "test"), transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Data loaded")

Data loaded


In [41]:
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    total_loss, correct, total = 0, 0, 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=loss.item())

    acc = 100 * correct / total
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}, Accuracy = {acc:.2f}%")

Epoch 1: 100%|██████████| 20/20 [02:58<00:00,  8.91s/it, loss=1.32]


Epoch 1: Loss = 28.0512, Accuracy = 55.22%


Epoch 2: 100%|██████████| 20/20 [02:54<00:00,  8.71s/it, loss=0.903]


Epoch 2: Loss = 20.0541, Accuracy = 66.29%


Epoch 3: 100%|██████████| 20/20 [03:01<00:00,  9.06s/it, loss=0.676]


Epoch 3: Loss = 14.6610, Accuracy = 73.68%


Epoch 4: 100%|██████████| 20/20 [02:54<00:00,  8.73s/it, loss=0.382]


Epoch 4: Loss = 10.2241, Accuracy = 82.66%


Epoch 5: 100%|██████████| 20/20 [02:55<00:00,  8.76s/it, loss=0.151]

Epoch 5: Loss = 8.2154, Accuracy = 86.52%


In [42]:
model.eval()
predictions = []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Predicting"):
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        predictions.extend(preds)

label_map = {v: k for k, v in train_dataset.class_to_idx.items()}
predicted_labels = [label_map[p] for p in predictions]

filenames = [path[0].split("/")[-1] for path in test_dataset.samples]
submission_df = pd.DataFrame({"filename": filenames, "predicted_label": predicted_labels})
submission_df.to_csv("phase1_predictions.csv", index=False)

print("Predictions saved to phase1_predictions.csv")

Predicting: 100%|██████████| 5/5 [00:14<00:00,  2.96s/it]

Predictions saved to phase1_predictions.csv


##phase 2


In [46]:
import zipfile

zip_path = "/content/drive/MyDrive/Copy of HV-AI-2025.zip"
extract_path = "/content/hv_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Re-extracted to:", extract_path)

Re-extracted to: /content/hv_dataset


In [47]:
import os

for root, dirs, files in os.walk("/content/hv_dataset"):
    for d in dirs:
        print("Folder:", os.path.join(root, d))

Folder: /content/hv_dataset/HV-AI-2025
Folder: /content/hv_dataset/split
Folder: /content/hv_dataset/HV-AI-2025/labeled_data
Folder: /content/hv_dataset/HV-AI-2025/unlabeled_data
Folder: /content/hv_dataset/HV-AI-2025/labeled_data/images
Folder: /content/hv_dataset/HV-AI-2025/unlabeled_data/images
Folder: /content/hv_dataset/split/train
Folder: /content/hv_dataset/split/test
Folder: /content/hv_dataset/split/train/cavallo
Folder: /content/hv_dataset/split/train/pecora
Folder: /content/hv_dataset/split/train/farfalla
Folder: /content/hv_dataset/split/train/gallina
Folder: /content/hv_dataset/split/train/scoiattolo
Folder: /content/hv_dataset/split/train/cane
Folder: /content/hv_dataset/split/train/elefante
Folder: /content/hv_dataset/split/train/gatto
Folder: /content/hv_dataset/split/train/ragno
Folder: /content/hv_dataset/split/train/mucca
Folder: /content/hv_dataset/split/test/cavallo
Folder: /content/hv_dataset/split/test/pecora
Folder: /content/hv_dataset/split/test/farfalla
Folder

In [48]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Transforms
common_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
train_dataset = datasets.ImageFolder("/content/hv_dataset/split/train", transform=common_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Class mapping
class_to_idx = train_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
num_classes = len(class_to_idx)

In [49]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

In [50]:
from PIL import Image
import glob
import os
from tqdm import tqdm

unlabeled_dir = "/content/hv_dataset/HV-AI-2025/unlabeled_data/images"
unlabeled_paths = sorted(glob.glob(os.path.join(unlabeled_dir, "*.*")))

pseudo_labels = []
pseudo_images = []

model.eval()
with torch.no_grad():
    for img_path in tqdm(unlabeled_paths, desc="Pseudo-labeling"):
        img = Image.open(img_path).convert("RGB")
        input_tensor = common_transform(img).unsqueeze(0).to(device)

        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        conf, pred = torch.max(probs, dim=1)

        if conf.item() >= 0.90:  # Confidence threshold
            pseudo_labels.append(idx_to_class[pred.item()])
            pseudo_images.append(img_path)


Pseudo-labeling: 100%|██████████| 14800/14800 [27:28<00:00,  8.98it/s]


In [51]:
import shutil

aug_data_path = "/content/hv_dataset/combined_train"
os.makedirs(aug_data_path, exist_ok=True)

# Copy original labeled images
for class_name in os.listdir("/content/hv_dataset/split/train"):
    os.makedirs(os.path.join(aug_data_path, class_name), exist_ok=True)
    class_dir = os.path.join("/content/hv_dataset/split/train", class_name)
    for f in os.listdir(class_dir):
        shutil.copy(os.path.join(class_dir, f), os.path.join(aug_data_path, class_name))

# Add pseudo-labeled images
for img_path, label in zip(pseudo_images, pseudo_labels):
    dest_dir = os.path.join(aug_data_path, label)
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy(img_path, dest_dir)

print("Combined labeled + pseudo-labeled dataset created")

Combined labeled + pseudo-labeled dataset created


In [52]:
aug_dataset = datasets.ImageFolder(aug_data_path, transform=common_transform)
aug_loader = DataLoader(aug_dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train for 3 epochs
for epoch in range(3):
    model.train()
    total_loss, correct, total = 0, 0, 0
    loop = tqdm(aug_loader, desc=f"[Retrain] Epoch {epoch+1}")

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item()

        loop.set_postfix(loss=loss.item())

    acc = 100 * correct / total
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}, Accuracy = {acc:.2f}%")

[Retrain] Epoch 1: 100%|██████████| 20/20 [03:03<00:00,  9.19s/it, loss=1.28]


Epoch 1: Loss = 25.3895, Accuracy = 60.35%


[Retrain] Epoch 2: 100%|██████████| 20/20 [02:55<00:00,  8.79s/it, loss=0.367]


Epoch 2: Loss = 13.2541, Accuracy = 78.01%


[Retrain] Epoch 3: 100%|██████████| 20/20 [02:53<00:00,  8.68s/it, loss=0.339]

Epoch 3: Loss = 5.4949, Accuracy = 90.69%


In [53]:
test_dataset = datasets.ImageFolder("/content/hv_dataset/split/test", transform=common_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model.eval()
predictions = []
filenames = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc=" Final Predictions"):
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        predictions.extend(preds)

filenames = [os.path.basename(path[0]) for path in test_dataset.samples]
predicted_labels = [idx_to_class[p] for p in predictions]

import pandas as pd
df = pd.DataFrame({"filename": filenames, "predicted_label": predicted_labels})
df.to_csv("phase2_predictions.csv", index=False)
print("Final predictions saved to phase2_predictions.csv")

Final Predictions: 100%|██████████| 5/5 [00:14<00:00,  2.90s/it]

Final predictions saved to phase2_predictions.csv


In [58]:
torch.save(model.state_dict(), "resnet18_phase2.pth")
print("Model saved as resnet18_phase2.pth")

Model saved as resnet18_phase2.pth
